# Phase 1 — Baseline Question Answering System

This notebook implements a minimal baseline Question Answering (QA) pipeline using a pretrained Large Language Model (LLM).

The purpose of this phase is to:
- establish a baseline QA behavior,
- observe response variability,
- and prepare for later reliability-oriented methods such as:
  - self-consistency,
  - calibration,
  - RAG,
  - and conformal prediction.

In [8]:
import os
os.getcwd()

'F:\\DriveD\\ML_Projects\\qa_uncertainty_project\\notebooks'

In [11]:
# ==========================================================
# IMPORT REQUIRED LIBRARIES
# ==========================================================

from transformers import pipeline

In [16]:
# ==========================================================
# INSTALL REQUIRED PACKAGES (RUN ONCE)
# ==========================================================

!pip install transformers torch

In [18]:
# ==========================================================
# IMPORT REQUIRED LIBRARIES
# ==========================================================

from transformers import AutoTokenizer
from transformers import AutoModelForSeq2SeqLM

In [20]:
# ==========================================================
# LOAD FLAN-T5 MODEL
# ==========================================================

model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


In [22]:
# ==========================================================
# VERIFY MODEL TYPE
# ==========================================================

print(model.config.model_type)

t5


In [24]:
# ==========================================================
# INITIAL GENERATION TESTS
# ==========================================================

inputs = tokenizer(
    "What is the coldest season?",
    return_tensors="pt"
)

outputs = model.generate(
    **inputs,
    max_length=50
)

print(
    tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )
)

winter


In [26]:
# ==========================================================
# BASELINE QUESTION ANSWERING FUNCTION
# ==========================================================

def qa_model(question):

    inputs = tokenizer(
        question,
        return_tensors="pt"
    )

    outputs = model.generate(
        **inputs,
        max_length=64
    )

    answer = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return answer

In [28]:
# ==========================================================
# TEST BASELINE QA MODEL
# ==========================================================

questions = [
    "Who is Albert Einstein?",
    "What is machine learning?",
    "What is Numerical Linear Algebra?"
]

for q in questions:

    print("=" * 50)
    print("Question:", q)
    print("Answer:", qa_model(q))

Question: Who is Albert Einstein?
Answer: physicist
Question: What is machine learning?
Answer: Machine learning is a computer science technique that uses computer algorithms to learn information about people and places.
Question: What is Numerical Linear Algebra?
Answer: linear algebra


## Note on Model Loading

Initial experiments attempted to use the HuggingFace `pipeline(...)` interface for text generation.

However, due to compatibility/runtime issues observed in the working environment, the final implementation adopted a direct approach using:

- `AutoTokenizer`
- `AutoModelForSeq2SeqLM`

This approach provided more stable and controllable generation behavior throughout the experiments.

# Baseline System Limitations

Initial experiments reveal several important limitations of the baseline QA system:

- responses may be overly short or vague,
- technical questions may produce inaccurate outputs,
- answers can vary significantly across generations,
- the model does not quantify uncertainty,
- and no mechanism exists to abstain from unreliable answers.

These observations motivate the development of reliability-oriented methods in the next experimental phases.

In [32]:
# ==========================================================
# MULTIPLE GENERATION EXPERIMENT
# ==========================================================

question = "What is Numerical Linear Algebra?"

for i in range(5):

    inputs = tokenizer(
        question,
        return_tensors="pt"
    )

    outputs = model.generate(
        **inputs,
        max_length=64,
        do_sample=True,
        temperature=0.8
    )

    answer = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    print(f"Sample {i+1}: {answer}")

Sample 1: linear algebra
Sample 2: linear algebra
Sample 3: a nonlinear algebra
Sample 4: algebra
Sample 5: linear algebra


# Observations on Response Variability

The generated answers may differ across stochastic generations, especially for technical or domain-specific questions.

This variability indicates that:
- LLM outputs are not always stable,
- confidence estimation is necessary,
- and aggregation strategies such as self-consistency may improve reliability.

This motivates the next phase of the project.